# Notebook — Entraînement ResNet-18 (pré-entraîné ImageNet) + Fine-tuning
## Projet MindVoice — Modalité Vision (Expressions faciales)

**Objectif :** entraîner un modèle de classification des émotions (8 classes) à partir de visages cropés.  
**Approche :** Transfer learning (ResNet-18 ImageNet) + fine-tuning sur notre dataset (~40k images train).



## 1. Imports + vérification PyTorch

In [2]:
import os
import time
import json
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import resnet18, ResNet18_Weights

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


Torch: 2.9.1+cpu
CUDA available: False


##  2. Configuration (chemins, hyperparamètres)

In [3]:
# =========================
# 1) CHEMINS (A MODIFIER)
# =========================
CSV_PATH = Path(r"C:/Users/khodj\Documents/M2_ISI/PFE/data_clean/meta/mindvoice_vision_labels_aug.csv")

# Dossier où sauvegarder le modèle + logs
RUN_DIR = Path(r"C:/Users/khodj/Documents/M2_ISI/PFE/resnet18_emotion")
RUN_DIR.mkdir(parents=True, exist_ok=True)

# =========================
# 2) HYPERPARAMETRES
# =========================
SEED = 42
NUM_EPOCHS = 15

BATCH_SIZE = 16           # si GPU: 64/128 ; si CPU: 16/32
NUM_WORKERS = 0           # Windows: 0 ou 2 selon stabilité
PIN_MEMORY = torch.cuda.is_available()

LR_HEAD = 1e-3            # lr pour la tête (classifier)
LR_BACKBONE = 1e-4        # lr plus petit pour le backbone
WEIGHT_DECAY = 1e-4

# Fine-tuning progressif :
EPOCHS_HEAD_ONLY = 2      # 2-3 epochs: on entraîne seulement la tête
UNFREEZE_AT_EPOCH = EPOCHS_HEAD_ONLY  # à partir de cet epoch, on dégèle (tout ou partie)

# Image size (doit matcher ton preprocessing : 224x224)
IMG_SIZE = 224

# Utilisation AMP (mixed precision) si GPU
USE_AMP = torch.cuda.is_available()

# =========================
# 3) LABELS
# =========================
# Tes émotions (adaptables)
EMOTIONS = ["neutral", "happy", "sad", "surprise", "fear", "anger", "disgust", "contempt"]

print("CSV:", CSV_PATH)
print("RUN_DIR:", RUN_DIR)
print("Emotions:", EMOTIONS)


CSV: C:\Users\khodj\Documents\M2_ISI\PFE\data_clean\meta\mindvoice_vision_labels_aug.csv
RUN_DIR: C:\Users\khodj\Documents\M2_ISI\PFE\resnet18_emotion
Emotions: ['neutral', 'happy', 'sad', 'surprise', 'fear', 'anger', 'disgust', 'contempt']


## 3. Seed + device

In [4]:
import random

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE


device(type='cpu')

## 4. Charger le CSV + vérifications

In [5]:
df = pd.read_csv(CSV_PATH)

# Colonnes attendues (au minimum)
required_cols = ["out_path", "emotion", "split"]
for c in required_cols:
    if c not in df.columns:
        raise ValueError(f"Colonne manquante dans le CSV: {c}")

# Nettoyage minimal
df["out_path"] = df["out_path"].astype(str)
df["emotion"] = df["emotion"].astype(str).str.lower().str.strip()
df["split"] = df["split"].astype(str).str.lower().str.strip()

# Filtrer les émotions non gérées
df = df[df["emotion"].isin(EMOTIONS)].copy()

# Vérifier l'existence des fichiers
exists_mask = df["out_path"].apply(lambda p: Path(p).exists())
missing = df[~exists_mask]
print("Fichiers manquants:", len(missing))
df = df[exists_mask].reset_index(drop=True)

print(df["split"].value_counts())
print(df["emotion"].value_counts().head(20))
df.head()


Fichiers manquants: 0
split
train    105408
val        4650
Name: count, dtype: int64
emotion
surprise    17333
happy       15497
anger       14831
disgust     13403
fear        13344
contempt    12858
sad         11914
neutral     10878
Name: count, dtype: int64


,id,out_path,emotion,split,source,parent_id,parent_path
0,affectnet_d214462e3e157f2d_0000000,C:\Users\khodj\Documents\M2_ISI\PFE\data_clean...,surprise,train,original,NaN,NaN
1,affectnet_e4477cd8bd3767f5_0000001,C:\Users\khodj\Documents\M2_ISI\PFE\data_clean...,anger,train,original,NaN,NaN
2,affectnet_840677ae10020230_0000002,C:\Users\khodj\Documents\M2_ISI\PFE\data_clean...,anger,val,original,NaN,NaN
3,affectnet_dda4086926d96610_0000003,C:\Users\khodj\Documents\M2_ISI\PFE\data_clean...,fear,train,original,NaN,NaN
4,affectnet_bcd8fd850bd42161_0000004,C:\Users\khodj\Documents\M2_ISI\PFE\data_clean...,anger,train,original,NaN,NaN


## 5. Encodage des labels

In [6]:
emotion_to_idx = {e:i for i,e in enumerate(EMOTIONS)}
idx_to_emotion = {i:e for e,i in emotion_to_idx.items()}

df["y"] = df["emotion"].map(emotion_to_idx).astype(int)

train_df = df[df["split"] == "train"].reset_index(drop=True)
val_df   = df[df["split"] == "val"].reset_index(drop=True)

print("Train:", len(train_df), "Val:", len(val_df))
train_df.head()


Train: 105408 Val: 4650


,id,out_path,emotion,split,source,parent_id,parent_path,y
0,affectnet_d214462e3e157f2d_0000000,C:\Users\khodj\Documents\M2_ISI\PFE\data_clean...,surprise,train,original,NaN,NaN,3
1,affectnet_e4477cd8bd3767f5_0000001,C:\Users\khodj\Documents\M2_ISI\PFE\data_clean...,anger,train,original,NaN,NaN,5
2,affectnet_dda4086926d96610_0000003,C:\Users\khodj\Documents\M2_ISI\PFE\data_clean...,fear,train,original,NaN,NaN,4
3,affectnet_bcd8fd850bd42161_0000004,C:\Users\khodj\Documents\M2_ISI\PFE\data_clean...,anger,train,original,NaN,NaN,5
4,affectnet_0a86499a26309331_0000005,C:\Users\khodj\Documents\M2_ISI\PFE\data_clean...,anger,train,original,NaN,NaN,5


## 6. Dataset PyTorch (lecture images)

In [7]:
from PIL import Image

class EmotionDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = row["out_path"]
        y = int(row["y"])

        # Lire image
        img = Image.open(img_path).convert("RGB")

        if self.transform:
            img = self.transform(img)

        return img, y, img_path


## 7. Transforms (train/val)

In [8]:
# Normalisation ImageNet (recommandée car le modèle est pré-entraîné ImageNet)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

val_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

train_ds = EmotionDataset(train_df, transform=train_tfms)
val_ds   = EmotionDataset(val_df, transform=val_tfms)

train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                      num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

val_dl   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                      num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

len(train_dl), len(val_dl)


(6588, 291)

## 8. Modèle : ResNet-18 pré-entraîné ImageNet + tête 8 classes

In [9]:
# Charger ResNet-18 avec poids ImageNet
weights = ResNet18_Weights.DEFAULT
model = resnet18(weights=weights)

# Remplacer la dernière couche (fc) par une couche adaptée à notre nb de classes
in_features = model.fc.in_features
model.fc = nn.Linear(in_features, len(EMOTIONS))

model = model.to(DEVICE)
print(model.fc)


Linear(in_features=512, out_features=8, bias=True)


## 9. Geler / dégeler (fine-tuning progressif)

In [10]:
def freeze_backbone(m):
    # Gèle tout sauf la tête fc
    for name, param in m.named_parameters():
        if not name.startswith("fc."):
            param.requires_grad = False
        else:
            param.requires_grad = True

def unfreeze_all(m):
    for param in m.parameters():
        param.requires_grad = True

# Au début: head only
freeze_backbone(model)

# Vérification
trainable = [n for n,p in model.named_parameters() if p.requires_grad]
print("Paramètres entraînables (début):", len(trainable))
trainable[:10]


Paramètres entraînables (début): 2


['fc.weight', 'fc.bias']

## 10. Optimiseur (LR différent backbone vs head)

In [11]:
def build_optimizer(m):
    # Paramètres head
    head_params = []
    backbone_params = []

    for name, param in m.named_parameters():
        if not param.requires_grad:
            continue
        if name.startswith("fc."):
            head_params.append(param)
        else:
            backbone_params.append(param)

    # Si backbone gelé => backbone_params vide
    param_groups = []
    if backbone_params:
        param_groups.append({"params": backbone_params, "lr": LR_BACKBONE})
    if head_params:
        param_groups.append({"params": head_params, "lr": LR_HEAD})

    opt = torch.optim.AdamW(param_groups, weight_decay=WEIGHT_DECAY)
    return opt

optimizer = build_optimizer(model)
criterion = nn.CrossEntropyLoss()

# Scheduler simple (optionnel, utile)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

print("Optimizer param groups:", len(optimizer.param_groups))
for i, g in enumerate(optimizer.param_groups):
    print(i, "lr=", g["lr"], "n_params=", sum(p.numel() for p in g["params"]))


Optimizer param groups: 1
0 lr= 0.001 n_params= 4104


C:\Users\khodj\AppData\Local\Temp\ipykernel_11068\790900973.py:30: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)


## 11. Métriques (accuracy, confusion matrix)

In [12]:
@torch.no_grad()
def evaluate(model, dataloader):
    model.eval()
    total = 0
    correct = 0
    loss_sum = 0.0

    all_preds = []
    all_targets = []

    for x, y, _ in dataloader:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        logits = model(x)
        loss = criterion(logits, y)

        preds = logits.argmax(dim=1)
        correct += (preds == y).sum().item()
        total += y.size(0)
        loss_sum += loss.item() * y.size(0)

        all_preds.append(preds.detach().cpu().numpy())
        all_targets.append(y.detach().cpu().numpy())

    all_preds = np.concatenate(all_preds) if all_preds else np.array([])
    all_targets = np.concatenate(all_targets) if all_targets else np.array([])

    acc = correct / max(1, total)
    avg_loss = loss_sum / max(1, total)
    return avg_loss, acc, all_targets, all_preds

def confusion_matrix_np(y_true, y_pred, n_classes):
    cm = np.zeros((n_classes, n_classes), dtype=int)
    for t, p in zip(y_true, y_pred):
        cm[int(t), int(p)] += 1
    return cm


## 12. Boucle d’entraînement (head-only puis fine-tuning)

In [ ]:
from tqdm.auto import tqdm
import time
import sys

def train_one_epoch(model, dataloader, optimizer, epoch):
    model.train()
    total = 0
    correct = 0
    loss_sum = 0.0

    print(f"\n--- Epoch {epoch+1}/{NUM_EPOCHS} ---", flush=True)

    pbar = tqdm(
        dataloader,
        total=len(dataloader),
        desc=f"Training",
        leave=True,
        dynamic_ncols=True,
        mininterval=0.1
    )

    for x, y, _ in pbar:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        if USE_AMP:
            with torch.cuda.amp.autocast():
                logits = model(x)
                loss = criterion(logits, y)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()

        preds = logits.argmax(dim=1)
        correct += (preds == y).sum().item()
        total += y.size(0)
        loss_sum += loss.item() * y.size(0)

        acc_running = correct / max(1, total)

        pbar.set_postfix({
            "loss": f"{loss.item():.4f}",
            "acc": f"{acc_running:.3f}"
        })

    acc = correct / max(1, total)
    avg_loss = loss_sum / max(1, total)
    return avg_loss, acc


best_val_acc = 0.0
best_path = RUN_DIR / "best_model.pt"
history = []

start_time = time.time()

for epoch in range(NUM_EPOCHS):

    if epoch == UNFREEZE_AT_EPOCH:
        print(f"\n🔓 Unfreeze backbone at epoch {epoch}", flush=True)
        unfreeze_all(model)
        optimizer = build_optimizer(model)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=NUM_EPOCHS - epoch
        )

    train_loss, train_acc = train_one_epoch(model, train_dl, optimizer, epoch)
    val_loss, val_acc, y_true, y_pred = evaluate(model, val_dl)

    scheduler.step()

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({
            "model_state": model.state_dict(),
            "emotion_to_idx": emotion_to_idx,
            "idx_to_emotion": idx_to_emotion,
            "img_size": IMG_SIZE,
            "imagenet_norm": {"mean": IMAGENET_MEAN, "std": IMAGENET_STD},
            "epoch": epoch,
            "val_acc": val_acc
        }, best_path)

    lr_list = [g["lr"] for g in optimizer.param_groups]
    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "val_loss": val_loss,
        "val_acc": val_acc,
        "lrs": lr_list,
    })

    print(
        f"✅ Epoch {epoch+1}/{NUM_EPOCHS} | "
        f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
        f"val_loss={val_loss:.4f} val_acc={val_acc:.4f} | "
        f"lr={lr_list[0]:.2e}",
        flush=True
    )

elapsed = time.time() - start_time
print(f"\n✅ Training terminé en {elapsed/60:.1f} minutes. Best val_acc={best_val_acc:.4f}")
print("Best model:", best_path)



--- Epoch 1/15 ---


Training:   0%|          | 0/6588 [00:00<?, ?it/s]

✅ Epoch 1/15 | train_loss=1.6893 train_acc=0.3695 | val_loss=1.6355 val_acc=0.4015 | lr=9.89e-04

--- Epoch 2/15 ---


Training:   0%|          | 0/6588 [00:00<?, ?it/s]

✅ Epoch 2/15 | train_loss=1.6350 train_acc=0.3948 | val_loss=1.5720 val_acc=0.4260 | lr=9.57e-04

🔓 Unfreeze backbone at epoch 2

--- Epoch 3/15 ---


...

Training:   0%|          | 0/6588 [00:00<?, ?it/s]

✅ Epoch 4/15 | train_loss=0.5132 train_acc=0.8124 | val_loss=0.8492 val_acc=0.7206 | lr=9.43e-05

--- Epoch 5/15 ---


Training:   0%|          | 0/6588 [00:00<?, ?it/s]

✅ Epoch 5/15 | train_loss=0.2945 train_acc=0.8940 | val_loss=0.9294 val_acc=0.7456 | lr=8.74e-05

--- Epoch 6/15 ---


Training:   0%|          | 0/6588 [00:00<?, ?it/s]

## 13. Sauvegarder l’historique + courbes

In [ ]:
import matplotlib.pyplot as plt


hist_path = RUN_DIR / "history.json"
with open(hist_path, "w", encoding="utf-8") as f:
    json.dump(history, f, ensure_ascii=False, indent=2)

print("History saved:", hist_path)

epochs = [h["epoch"] for h in history]
train_accs = [h["train_acc"] for h in history]
val_accs   = [h["val_acc"] for h in history]
train_losses = [h["train_loss"] for h in history]
val_losses   = [h["val_loss"] for h in history]

plt.figure()
plt.plot(epochs, train_accs, label="train_acc")
plt.plot(epochs, val_accs, label="val_acc")
plt.xlabel("epoch")
plt.ylabel("accuracy")
plt.legend()
plt.title("Accuracy")
plt.show()

plt.figure()
plt.plot(epochs, train_losses, label="train_loss")
plt.plot(epochs, val_losses, label="val_loss")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.legend()
plt.title("Loss")
plt.show()


NameError: name 'history' is not defined

## 14. Confusion matrix sur validation (lisible)

In [1]:
# Charger le meilleur modèle puis évaluer
ckpt = torch.load(best_path, map_location=DEVICE)
model.load_state_dict(ckpt["model_state"])
model.eval()

val_loss, val_acc, y_true, y_pred = evaluate(model, val_dl)
print("Best model val_acc:", val_acc)

cm = confusion_matrix_np(y_true, y_pred, n_classes=len(EMOTIONS))

# Affichage simple
plt.figure(figsize=(8, 6))
plt.imshow(cm, interpolation="nearest")
plt.title("Confusion Matrix (val)")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.xticks(np.arange(len(EMOTIONS)), EMOTIONS, rotation=45, ha="right")
plt.yticks(np.arange(len(EMOTIONS)), EMOTIONS)
plt.colorbar()
plt.tight_layout()
plt.show()

# Option: normaliser
cm_norm = cm / np.maximum(1, cm.sum(axis=1, keepdims=True))
plt.figure(figsize=(8, 6))
plt.imshow(cm_norm, interpolation="nearest")
plt.title("Confusion Matrix Normalized (val)")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.xticks(np.arange(len(EMOTIONS)), EMOTIONS, rotation=45, ha="right")
plt.yticks(np.arange(len(EMOTIONS)), EMOTIONS)
plt.colorbar()
plt.tight_layout()
plt.show()


NameError: name 'torch' is not defined

## 15. Fonction d’inférence sur une image (préparation webcam/vidéo)

In [ ]:
from PIL import Image

@torch.no_grad()
def predict_image(model, img_path):
    model.eval()
    img = Image.open(img_path).convert("RGB")

    tfm = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])

    x = tfm(img).unsqueeze(0).to(DEVICE)
    logits = model(x)
    probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
    pred_idx = int(np.argmax(probs))
    return idx_to_emotion[pred_idx], probs

# Test sur une image val
sample_path = val_df.sample(1, random_state=1)["out_path"].iloc[0]
pred, probs = predict_image(model, sample_path)
print("Image:", sample_path)
print("Pred:", pred)
print("Top3:", sorted([(EMOTIONS[i], float(probs[i])) for i in range(len(EMOTIONS))],
                     key=lambda x: x[1], reverse=True)[:3])


## 16. Inférence vidéo (découpage frames + crop visage + lissage)

In [ ]:
import cv2
from collections import deque

# === A CONFIGURER ===
VIDEO_PATH = r"/chemin/vers/video.mp4"
SAMPLE_FPS = 8            # traite ~8 images/seconde
SMOOTH_WINDOW = 8         # moyenne glissante sur 1 sec si 8 fps
FACE_MARGIN = 0.25

# Haar (simple offline)
haar_path = cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
face_cascade = cv2.CascadeClassifier(haar_path)

def detect_face_bbox_bgr(frame_bgr):
    gray = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, 1.1, 5, minSize=(40, 40))
    if len(faces) == 0:
        return None
    faces = sorted(faces, key=lambda b: b[2]*b[3], reverse=True)
    return faces[0]

def crop_with_margin_bgr(frame_bgr, bbox, margin=0.25):
    h, w = frame_bgr.shape[:2]
    x, y, bw, bh = bbox
    mx = int(bw * margin)
    my = int(bh * margin)
    x1 = max(0, x - mx); y1 = max(0, y - my)
    x2 = min(w, x + bw + mx); y2 = min(h, y + bh + my)
    return frame_bgr[y1:y2, x1:x2]

def preprocess_frame_for_model(face_bgr):
    # BGR -> RGB -> PIL -> tensor norm
    face_rgb = cv2.cvtColor(face_bgr, cv2.COLOR_BGR2RGB)
    pil = Image.fromarray(face_rgb)
    x = val_tfms(pil).unsqueeze(0)  # val_tfms fait resize + normalize
    return x

@torch.no_grad()
def run_video_inference(model, video_path):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError("Impossible d'ouvrir la vidéo")

    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps <= 0:
        fps = 25.0

    # échantillonnage: 1 frame sur N
    step = max(1, int(round(fps / SAMPLE_FPS)))

    probs_buffer = deque(maxlen=SMOOTH_WINDOW)
    timeline = []  # [(t, pred, probs)]

    frame_idx = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if frame_idx % step != 0:
            frame_idx += 1
            continue

        t_sec = frame_idx / fps

        bbox = detect_face_bbox_bgr(frame)
        if bbox is None:
            # pas de visage => unknown
            timeline.append((t_sec, "unknown", None))
            frame_idx += 1
            continue

        face = crop_with_margin_bgr(frame, bbox, margin=FACE_MARGIN)
        x = preprocess_frame_for_model(face).to(DEVICE)

        logits = model(x)
        probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
        probs_buffer.append(probs)

        # lissage
        smooth_probs = np.mean(np.stack(list(probs_buffer)), axis=0)
        pred_idx = int(np.argmax(smooth_probs))
        pred = idx_to_emotion[pred_idx]

        timeline.append((t_sec, pred, smooth_probs))
        frame_idx += 1

    cap.release()
    return timeline

timeline = run_video_inference(model, VIDEO_PATH)
print("Timeline length:", len(timeline))
print("Exemples:", timeline[:5])


## 17. Export résultats vidéo (JSON)

In [ ]:
out_json = RUN_DIR / "video_predictions.json"

serializable = []
for t, pred, probs in timeline:
    if probs is None:
        serializable.append({"t": float(t), "pred": pred, "probs": None})
    else:
        serializable.append({
            "t": float(t),
            "pred": pred,
            "probs": {EMOTIONS[i]: float(probs[i]) for i in range(len(EMOTIONS))}
        })

with open(out_json, "w", encoding="utf-8") as f:
    json.dump(serializable, f, ensure_ascii=False, indent=2)

print("✅ Export vidéo:", out_json)
